In [3]:
import pandas as pd
import numpy as np
import clip
import torch
import tqdm
import json
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

In [4]:
data = pd.read_csv('data/datasets/train-sarcasm-dataset.csv', encoding='utf-8')
data.head()

,Image,Caption,Label
0,8ae451edcd8ebf697f8763ece249115813149c55733bf8...,Cô ấy trên mạng vs cô ấy ngoài đời =))),sarcasm
1,35370ffd6c791d6f8c4ab3dd4363ed468fab41e4824ee9...,Người tâm linh giao tiếp với người thực tế :))),not-sarcasm
2,316fdd1477725b9fb1a55015ac06b68b92b50bd4303e08...,Hình như Trăng hôm nay đẹp quá mọi người ạ! 😃 ...,sarcasm
3,8a0f34e0e30e4e5cfb306933c1d25fa801a5da78646b59...,MỌI NGƯỜI NGHĨ SAO VỀ PHÁT BIỂU CỦA SHARK VIỆT...,not-sarcasm
4,e517a5e95d1065886a7c815e82fe254381d4f9f4b244d4...,2 tay hai nàng chứ việc gì phải lệ hai hàng,sarcasm


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
labels = ["sarcasm", "not-sarcasm"]
text_inputs = processor(text=[f"A photo that is {label}." for label in labels], return_tensors="pt", padding=True).to(device)

c:\Users\ADMIN\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
text_inputs

{'input_ids': tensor([[49406,   320,  1125,   682,   533, 37246,   269, 49407, 49407, 49407],
        [49406,   320,  1125,   682,   533,   783,   268, 37246,   269, 49407]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [7]:
from torch.utils.data import random_split

train_size = int(0.7 * len(data))
val_size = len(data) - train_size
train_dataset, val_dataset = random_split(data, [train_size, val_size])

In [8]:
print("Lenght of train data: {}".format(len(train_dataset)))
print("Lenght of val data: {}".format(len(val_dataset)))

Lenght of train data: 7563
Lenght of val data: 3242


In [9]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

# Define a custom dataset class
class ImageDataset(Dataset):
    def __init__(self, data, subcategories):
        self.data = data
        self.subcategories = subcategories
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_path = "data/train-images/{}".format(item['Image'])
        image = Image.open(image_path).convert("RGB")  
        subcategory = item['Label']
        label = self.subcategories.index(subcategory) if subcategory in self.subcategories else -1
        return self.transform(image), label


In [10]:
from torch.utils.data import DataLoader

subcategories = list(data['Label'].unique())
subcategories

['sarcasm', 'not-sarcasm']

In [43]:
train_dataset, val_dataset = train_test_split(data.to_dict(orient='records'), test_size=0.2, random_state=42)
train_loader = DataLoader(ImageDataset(train_dataset, subcategories), batch_size=32, shuffle=True)
val_loader = DataLoader(ImageDataset(val_dataset, subcategories), batch_size=32, shuffle=False)

In [44]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [45]:
import torch.nn as nn

class CLIPFineTuner(nn.Module):
    def __init__(self, clip_model, num_classes):
        super(CLIPFineTuner, self).__init__()
        self.clip_model = clip_model
        self.fc = nn.Linear(clip_model.config.projection_dim, num_classes)

    def forward(self, images, input_ids):
        outputs = self.clip_model(pixel_values=images, input_ids=input_ids)
        logits = self.fc(outputs.image_embeds)
        return logits

num_classes = len(subcategories)
model_ft = CLIPFineTuner(model, num_classes).to(device)

In [46]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ft.fc.parameters(), lr=1e-4)

In [47]:
from tqdm import tqdm

num_epochs = 4

for epoch in range(num_epochs):
    model_ft.train() 
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}, Loss: 0.0000")

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)  
        optimizer.zero_grad()  
        input_ids = text_inputs.input_ids.repeat(images.size(0), 1)  
        outputs = model_ft(images, input_ids) 
        loss = criterion(outputs, labels)
        loss.backward()  
        optimizer.step()  

        running_loss += loss.item() 
        pbar.set_description(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}") 

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}') 

    model_ft.eval()  
    correct = 0 
    total = 0 

    with torch.no_grad():  
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device) 
            input_ids = text_inputs.input_ids.repeat(images.size(0), 1) 
            outputs = model_ft(images, input_ids)  
            _, predicted = torch.max(outputs.data, 1)  
            total += labels.size(0) 
            correct += (predicted == labels).sum().item()  
    print(f'Validation Accuracy: {100 * correct / total}%') 

# Lưu mô hình đã fine-tuned
torch.save(model_ft.state_dict(), 'clip_finetuned.pth')  


Epoch 1/4, Loss: 0.0000:   0%|          | 0/271 [00:00<?, ?it/s]c:\Users\ADMIN\anaconda3\Lib\site-packages\transformers\models\clip\modeling_clip.py:480: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Epoch 1/4, Loss: 0.6838: 100%|██████████| 271/271 [02:40<00:00,  1.69it/s]


Epoch [1/4], Loss: 0.6838
Validation Accuracy: 57.380842202683944%


Epoch 2/4, Loss: 0.6724: 100%|██████████| 271/271 [01:47<00:00,  2.53it/s]


Epoch [2/4], Loss: 0.6724
Validation Accuracy: 59.78713558537714%


Epoch 3/4, Loss: 0.6638: 100%|██████████| 271/271 [01:47<00:00,  2.51it/s]


Epoch [3/4], Loss: 0.6638
Validation Accuracy: 61.22165664044424%


Epoch 4/4, Loss: 0.6570: 100%|██████████| 271/271 [01:54<00:00,  2.37it/s]


Epoch [4/4], Loss: 0.6570
Validation Accuracy: 63.026376677464135%


In [11]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn

class CLIPFineTuner(nn.Module):
    def __init__(self, clip_model, num_classes):
        super(CLIPFineTuner, self).__init__()
        self.clip_model = clip_model
        self.fc = nn.Linear(clip_model.config.projection_dim, num_classes)

    def forward(self, images, input_ids):
        outputs = self.clip_model(pixel_values=images, input_ids=input_ids)
        logits = self.fc(outputs.image_embeds)
        return logits
    
num_classes = 2  # Assuming 2 classes: sarcasm, not-sarcasm
model_ft = CLIPFineTuner(model, num_classes).to(device)
model_ft.load_state_dict(torch.load('clip_finetuned.pth', map_location=device))

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_146048\3599255540.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ft.load_state_dict(torch.load('clip_finetuned.pth', 

<All keys matched successfully>

In [12]:
model_ft.eval()

CLIPFineTuner(
  (clip_model): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPSdpaAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bia

In [13]:
labels = ["sarcasm", "not-sarcasm"]
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


In [24]:
def predict_sarcasm(image_path):
    # Open and preprocess the image
    image = Image.open(image_path).convert("RGB")

    # Preprocess image for the CLIP model
    image_input = processor(images=image, return_tensors="pt").to(device)

    # Create input text prompts
    text_inputs = processor(text=[f"A photo that is {label}." for label in labels], return_tensors="pt", padding=True).to(device)
    # Disable gradient calculation (not necessary for inference)
    with torch.no_grad():
        # Get predictions from the model
        outputs = model_ft(image_input.pixel_values, text_inputs.input_ids)
        probabilities = torch.softmax(outputs, dim=-1)
        print(probabilities)
        predicted_class_idx = torch.argmax(probabilities, dim=-1).item()

    # Return the predicted label and probability
    return labels[predicted_class_idx], probabilities[0][predicted_class_idx].item()

In [19]:
import json
import numpy as np
from PIL import Image

def load_annotation(file_path):
    data = []

    with open (file_path, 'r', encoding="utf-8") as file:
        data = json.load(file)    
    return data

annotation_path = "data/vimmsd-public-test.json"
data = load_annotation(annotation_path)

In [18]:
# image_path = "data/train-images/{}".format(item['Image'])
image_path = "data/dev-images/{}".format("0a8b18d7013f7bd4a11ee5e6f94d7327387eddf9c5fee61b5acdc5b254dc6fdb.jpg")
predicted_label, confidence = predict_sarcasm(image_path)
print(f"Predicted: {predicted_label}, Confidence: {confidence:.4f}")

c:\Users\ADMIN\anaconda3\Lib\site-packages\transformers\models\clip\modeling_clip.py:480: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Predicted: sarcasm, Confidence: 0.5444


In [26]:
def map_test_label(label):
    return label if (label == "not-sarcasm") else "sarcasm"

labels_test = []
export = []

def map_label(label):
    return 1 if (label== "sarcasm") else 0

for id in data:
    image_name = data[id]['image']
    label_test = map_test_label(data[id]['label'])

    image_path = "data/dev-images/{}".format(image_name)
    predicted_label, confidence = predict_sarcasm(image_path)
    labels_test.append(map_label(label_test))

    export.append([image_name, predicted_label, np.round(confidence, 4)])

tensor([[0.5248, 0.4752]], device='cuda:0')
tensor([[0.5249, 0.4751]], device='cuda:0')
tensor([[0.4812, 0.5188]], device='cuda:0')
tensor([[0.4685, 0.5315]], device='cuda:0')
tensor([[0.4157, 0.5843]], device='cuda:0')
tensor([[0.4436, 0.5564]], device='cuda:0')
tensor([[0.4490, 0.5510]], device='cuda:0')
tensor([[0.4098, 0.5902]], device='cuda:0')
tensor([[0.5247, 0.4753]], device='cuda:0')
tensor([[0.5348, 0.4652]], device='cuda:0')
tensor([[0.5224, 0.4776]], device='cuda:0')
tensor([[0.4498, 0.5502]], device='cuda:0')
tensor([[0.4940, 0.5060]], device='cuda:0')
tensor([[0.4772, 0.5228]], device='cuda:0')
tensor([[0.5304, 0.4696]], device='cuda:0')
tensor([[0.4710, 0.5290]], device='cuda:0')
tensor([[0.3944, 0.6056]], device='cuda:0')
tensor([[0.4222, 0.5778]], device='cuda:0')
tensor([[0.3946, 0.6054]], device='cuda:0')
tensor([[0.4050, 0.5950]], device='cuda:0')
tensor([[0.4873, 0.5127]], device='cuda:0')
tensor([[0.5113, 0.4887]], device='cuda:0')
tensor([[0.4816, 0.5184]], devic

In [21]:
export_df = pd.DataFrame(export)
export_df.to_csv("data/exports/public-test-img-clip-labels-v0.1.csv", index=None, header=None)

In [27]:
labels_test

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
